In [2]:
from fastapi import FastAPI
import pandas as pd
import os
import datetime as dt

# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
# Datensatz aufbereiten
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

# import Datensatz
#___________________________________
df_raw = pd.read_csv("Gesamtdatensatz.csv")

# relevante Spalten behalten
#___________________________________
df_copy = df_raw.copy(deep=False)
df_pedData = df_copy[['timestamp', 'location_name',
                    'weather_condition', 'temperature',
                    'pedestrians_count', 'ltr_pedestrians_count',
                    'rtl_pedestrians_count', 'adult_ltr_pedestrians_count',
                    'adult_rtl_pedestrians_count', 'child_rtl_pedestrians_count',
                    'child_ltr_pedestrians_count']]

image_paths = ["src/assets/clear-day.png","src/assets/clear-night.png", "src/assets/cloudy.png", "src/assets/fog.png", "src/assets/partly-cloudy-day.png", "src/assets/partly-cloudy-night.png", "src/assets/rain.png", "src/assets/snow.png"]

# zusätzliche Spalten erzeugen
#___________________________________
df_pedData['weather_icon'] = df_copy['weather_condition'].map({
    os.path.splitext(os.path.basename(p))[0]: p
    for p in image_paths
})
df_pedData['timestamp'] = pd.to_datetime(df_pedData['timestamp'])
print(df_pedData.dtypes)
df_pedData['date'] = df_pedData['timestamp'].dt.date
df_pedData['hour'] = df_pedData['timestamp'].dt.hour
df_pedData['pedestrian_grey'] = df_pedData[['ltr_pedestrians_count', 'rtl_pedestrians_count']].min(axis=1)
df_pedData['pedestrian_diff'] = ((df_pedData['ltr_pedestrians_count'] - df_pedData['rtl_pedestrians_count'])**2)**0.5
df_pedData['max_val'] = (
    df_pedData.groupby(['date'])[['ltr_pedestrians_count', 'rtl_pedestrians_count']]
      .transform('max')      # max per column per date
      .max(axis=1)           # max across the two columns
)+20

timestamp                      datetime64[ns, UTC]
location_name                               object
weather_condition                           object
temperature                                float64
pedestrians_count                            int64
ltr_pedestrians_count                        int64
rtl_pedestrians_count                        int64
adult_ltr_pedestrians_count                  int64
adult_rtl_pedestrians_count                  int64
child_rtl_pedestrians_count                  int64
child_ltr_pedestrians_count                  int64
weather_icon                                object
dtype: object


C:\Users\jonat\AppData\Local\Temp\ipykernel_3016\3352916799.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pedData['weather_icon'] = df_copy['weather_condition'].map({
C:\Users\jonat\AppData\Local\Temp\ipykernel_3016\3352916799.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pedData['timestamp'] = pd.to_datetime(df_pedData['timestamp'])
C:\Users\jonat\AppData\Local\Temp\ipykernel_3016\3352916799.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram

In [10]:
df = pd.read_csv("Gesamtdatensatz.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date

In [42]:
df_aggregate = df[df["collection_type"] == "measured"]
df_aggregate = df_aggregate.drop_duplicates(subset=["date", "location_name"])
df_filter = df_aggregate[["date", "location_name"]]
df_filter




,date,location_name
0,2021-09-28,Bahnhofstrasse (Mitte)
1,2021-09-28,Bahnhofstrasse (Nord)
2,2021-09-28,Bahnhofstrasse (Süd)
8,2021-09-29,Bahnhofstrasse (Mitte)
9,2021-09-29,Bahnhofstrasse (Nord)
...,...,...
134315,2025-07-29,Lintheschergasse
134408,2025-07-30,Bahnhofstrasse (Mitte)
134409,2025-07-30,Bahnhofstrasse (Nord)
134410,2025-07-30,Bahnhofstrasse (Süd)
